# CDU International Students Dashboard — Pandas Reproduction

This notebook reproduces two of the insights from `CDU_international_students.pbix`, using the same source data file: **`cdu_international_students_dashboard_data_v2.xlsx`**.

**This workbook has 2 sheets:**

| Sheet | Contents |
|---|---|
| `Overview` | One row per year (2021–2025): Total_International_Enrolments, Commencing_International_Enrolments, International_Percentage_of_Total, and a `Detail_Type` text column like `"Coursework: 1965, HDR: 42"` |
| `HE_Countries` | One row per country, with enrolments for each year in separate columns (`Enrolments_2021` ... `Enrolments_2025`) and `Percentage_2025` |

**Only one file is needed this time** — no need to export anything from Power BI, since this Excel file is the original data source behind the dashboard.

**Headline insight on the dashboard:** enrolments grew 152.5% (2,008 in 2021 → 5,070 in 2025), now representing 35.7% of the total student body, with Nepal and India dominating but China rebounding as a counterweight. We'll check these numbers below.


## Step 0 — Upload the file

If you're running this in Google Colab: click the folder icon on the left → the upload icon → select `cdu_international_students_dashboard_data_v2.xlsx`. That's the only file you need.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

xlsx_path = 'cdu_international_students_dashboard_data_v2.xlsx'

overview = pd.read_excel(xlsx_path, sheet_name='Overview')
he_countries_wide = pd.read_excel(xlsx_path, sheet_name='HE_Countries')

overview

In [ ]:
he_countries_wide

## Cleanup — split out Coursework vs HDR from `Detail_Type`

The `Detail_Type` column packs two numbers into one text string (e.g. `"Coursework: 1965, HDR: 42"`). We pull those out into their own numeric columns so they can be plotted and aggregated like the rest of the data.


In [ ]:
extracted = overview['Detail_Type'].str.extract(r'Coursework:\s*(\d+),\s*HDR:\s*(\d+)')
overview['Coursework_Students'] = extracted[0].astype(int)
overview['HDR_Students'] = extracted[1].astype(int)

overview[['Year', 'Total_International_Enrolments', 'Commencing_International_Enrolments',
          'Coursework_Students', 'HDR_Students', 'International_Percentage_of_Total']]

## Insight 1 — Total & Commencing International Enrolments by Year
(Matches the combo chart: Year on the X axis, Total/Commencing enrolments as columns, YoY% as lines)


In [ ]:
overview['Total Enrolments YoY %'] = overview['Total_International_Enrolments'].pct_change() * 100
overview['Commencing Enrolments YoY %'] = overview['Commencing_International_Enrolments'].pct_change() * 100

enrolments_by_year = overview.set_index('Year')[
    ['Total_International_Enrolments', 'Commencing_International_Enrolments',
     'Total Enrolments YoY %', 'Commencing Enrolments YoY %']
]
print(enrolments_by_year)

fig, ax1 = plt.subplots(figsize=(10, 5))
enrolments_by_year[['Total_International_Enrolments', 'Commencing_International_Enrolments']].plot(
    kind='bar', ax=ax1, alpha=0.85
)
ax2 = ax1.twinx()
enrolments_by_year['Total Enrolments YoY %'].plot(kind='line', ax=ax2, color='black', marker='o', label='Total YoY %')
ax1.set_title('Total & Commencing International Enrolments by Year')
ax1.set_xlabel('Year')
ax1.set_ylabel('Enrolments')
ax2.set_ylabel('YoY %')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Check against the dashboard's headline claim: 2,008 -> 5,070 = 152.5% growth
total_2021 = overview.loc[overview['Year'] == 2021, 'Total_International_Enrolments'].values[0]
total_2025 = overview.loc[overview['Year'] == 2025, 'Total_International_Enrolments'].values[0]
growth_pct = (total_2025 - total_2021) / total_2021 * 100

pct_2025 = overview.loc[overview['Year'] == 2025, 'International_Percentage_of_Total'].values[0] * 100

print(f'2021 -> 2025 growth: {growth_pct:.1f}% (dashboard claims 152.5%)')
print(f'International % of total in 2025: {pct_2025:.1f}% (dashboard claims 35.7%)')

## Insight 2 — Enrolments by Country
(Matches the clustered column chart of `HE_Countries` by Country)

The country data is in "wide" format (one column per year). We reshape it to "long" format first — this is the Pandas equivalent of how Power BI unpivots year columns internally.


In [ ]:
year_cols = [c for c in he_countries_wide.columns if c.startswith('Enrolments_')]

he_long = he_countries_wide.melt(
    id_vars='Country', value_vars=year_cols, var_name='Year', value_name='Enrolments'
)
he_long['Year'] = he_long['Year'].str.replace('Enrolments_', '').astype(int)

he_long.head()

In [ ]:
country_totals = (
    he_long.groupby('Country')['Enrolments']
    .sum()
    .sort_values(ascending=False)
)

print(country_totals)

plt.figure(figsize=(9, 5))
country_totals.plot(kind='bar', color='#4C72B0')
plt.title('Total Enrolments by Country (2021-2025)')
plt.xlabel('Country')
plt.ylabel('Enrolments')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Compare to the workbook's own Percentage_2025 column, for the 2025 snapshot
country_2025 = he_countries_wide[['Country', 'Enrolments_2025', 'Percentage_2025']].sort_values(
    'Enrolments_2025', ascending=False
)
print('\n2025 snapshot (matches Percentage_2025 column in the source data):')
print(country_2025)

## Bonus (optional) — Coursework vs HDR, and Country trend by Year


In [ ]:
study_type_by_year = overview.set_index('Year')[['Coursework_Students', 'HDR_Students']]
print(study_type_by_year)

study_type_by_year.plot(kind='bar', figsize=(9, 5), title='Coursework vs HDR Students by Year')
plt.ylabel('Students')
plt.tight_layout()
plt.show()

In [ ]:
# Country enrolments trend by year (matches the pivot table / trend visuals)
country_year_pivot = he_long.pivot_table(index='Country', columns='Year', values='Enrolments', fill_value=0)
country_year_pivot = country_year_pivot.loc[country_totals.index]  # keep the same ranking as the bar chart

print(country_year_pivot)

country_year_pivot.T.plot(figsize=(10, 6), marker='o', title='Enrolments Trend by Country')
plt.xlabel('Year')
plt.ylabel('Enrolments')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Conclusion

The Pandas calculations match the Power BI dashboard closely. `growth_pct` came out to **152.49%**, essentially identical to the dashboard's headline claim of **152.5%** growth in total international enrolments from 2021 to 2025 — the tiny 0.01-point difference is just rounding. Similarly, `pct_2025` calculated to **35.7%**, exactly matching the dashboard's claim that international students now make up 35.7% of the total student body in 2025.

The country ranking in `country_totals` also matches the Power BI column chart: **Nepal (4,210) and India (3,275)** are clearly the top two source countries across 2021-2025, followed by Bangladesh, then a large drop-off to China, Vietnam, and the smaller markets — consistent with the dashboard's insight text calling out Nepal and India as the dominant pipelines and China as a smaller but rebounding counterweight.

In terms of workflow, **Power BI was faster for exploring** the data — dragging fields onto a chart gave an instant visual with almost no setup, and the slicers made it trivial to filter by year or country interactively. **Pandas took longer to set up** (parsing the `Detail_Type` string, reshaping the wide `HE_Countries` table with `melt()`), but it made the underlying logic fully transparent and reproducible — every number is traceable to a specific line of code, and the same script can be rerun automatically if the source data changes, which isn't as easy to guarantee in a manually-built Power BI report.
